In [17]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import itertools

from seq_utils import generate_kv_mapping

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
seq_folder = "/Users/ccnlab/Development/sequences/shaping/trans"

NUM_KEYS = 4
NUM_FOOD = 4
NUM_SHOPS = 4
ALL_VF_IMAGES_SET = 4
TRANS_FOLDER = 1
OUTPUT_COL_ORDER = [
    "stim",
    "correct_key",
    "block",
    "img_folder",
    "trans_folder",
    "key0_trans",
    "key1_trans",
    "key2_trans",
    "key3_trans",
    "shop0_food",
    "shop1_food",
    "shop2_food",
    "shop3_food",
    "trans0_shop",
    "trans1_shop",
    "trans2_shop",
    "trans3_shop",
    "set_size",
]


In [19]:
data = pd.read_csv(f'{seq_folder}/shaping_trans0_learning.csv')
data

,stim,correct_key,block,img_folder,trans_folder,key0_trans,key1_trans,key2_trans,key3_trans,shop0_food,shop1_food,shop2_food,shop3_food,trans0_shop,trans1_shop,trans2_shop,trans3_shop,set_size
0,0,0,1,1,1,2,0,1,3,1,2,3,0,1,0,3,2,6
1,2,2,1,1,1,1,0,3,2,3,0,2,1,1,0,3,2,6
2,3,1,1,1,1,2,1,0,3,3,2,0,1,1,0,3,2,6
3,1,3,1,1,1,3,2,1,0,2,1,0,3,1,0,3,2,6
4,0,3,1,1,1,3,0,1,2,2,3,1,0,1,0,3,2,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,0,1,2,1,1,1,3,0,2,2,1,0,3,1,0,3,2,6
152,4,0,2,1,1,3,1,0,2,0,2,1,3,1,0,3,2,6
153,3,2,2,1,1,2,3,0,1,0,3,2,1,1,0,3,2,6
154,5,1,2,1,1,0,2,3,1,1,2,0,3,1,0,3,2,6


In [ ]:
# trans to shop mapping
trans_to_shop = {}
for i in np.arange(4):
    trans_to_shop[i] = data[f'trans{i}_shop'].iloc[0]

# stim to food mapping
nonshaped_block = data[data.block == 2]
stim_to_food = {}
for stim in nonshaped_block.stim.unique():
    row = nonshaped_block[nonshaped_block.stim == stim].iloc[0]
    k = row.correct_key
    correct_shop = trans_to_shop[row[f'key{k}_trans']]
    stim_to_food[row.stim] = row[f'shop{correct_shop}_food']

stim_to_food

{1: 1, 0: 0, 5: 3, 4: 1, 2: 2, 3: 3}

In [56]:
def generate_blocked_seq(num_stims, num_shops, num_rep_per_stim):
    # 1. Create all num_stims unique (stim, shop) pairs
    seq_order = np.random.permutation(num_stims)
    stim_seq = np.repeat(seq_order, num_rep_per_stim)

    shop_order = np.random.permutation(num_shops)
    shop_second_order = np.random.choice(
        num_shops, size=num_stims - num_shops, replace=True
    )
    shop_order = np.concatenate([shop_order, shop_second_order])

    stim_to_shop = {stim: shop for stim, shop in zip(seq_order, shop_order)}
    shop_seq = np.array([stim_to_shop[stim] for stim in stim_seq])
    return pd.DataFrame({"stim": stim_seq, "shop": shop_seq})  # dict → columns


# generate sequence of stim and shop pairs
TOTAL_ITER = 8
iter_seq = {}
for first_iter in [4, 6, 8]:
    combined_seq = generate_blocked_seq(
        len(stim_to_food.keys()), NUM_SHOPS, first_iter
    )
    second_iter = TOTAL_ITER - first_iter
    if second_iter > 0:
        second_seq = generate_blocked_seq(
            len(stim_to_food.keys()), NUM_SHOPS, second_iter
        )
        combined_seq = pd.concat([combined_seq, second_seq])
    print(combined_seq.shape)
    iter_seq[first_iter] = combined_seq

(48, 2)
(48, 2)
(48, 2)


In [60]:
SAME_COL = [
    "block",
    "img_folder",
    "trans_folder",
    "set_size",
    # mapping from key to trans
    "key0_trans",
    "key1_trans",
    "key2_trans",
    "key3_trans",
    # mapping from trans to shop
    "trans0_shop",
    "trans1_shop",
    "trans2_shop",
    "trans3_shop",
]

def generate_shaping_blocked_seq(first_block, stim_to_food, shop_to_trans, stim_shop_seq):
    previous_stim = -1
    blocked_seq = {k: [] for k in OUTPUT_COL_ORDER}
    food_order = []
    # place food on top of the shop and get correct key by shop -> trans -> key
    for i, row in first_block.iterrows():
        seq_row = stim_shop_seq.iloc[i]
        stim = seq_row.stim
        shop = seq_row.shop
        if previous_stim != stim:
            food_order = np.random.permutation(NUM_FOOD)
            # swap the food based on correct shop
            fav_food_idx = np.where(food_order == stim_to_food[stim])[0][0]
            food_order[fav_food_idx], food_order[shop] = (
                food_order[shop],
                food_order[fav_food_idx],
            )

            #print(food_order, shop, stim_to_food[stim])
            previous_stim = stim

        for col in SAME_COL:
            blocked_seq[col].append(row[col])

        blocked_seq["stim"].append(stim)
        for i, food in enumerate(food_order):
            blocked_seq[f"shop{i}_food"].append(food)

        correct_trans = shop_to_trans[shop]
        correct_key = next(
            k for k in range(NUM_KEYS) if row[f"key{k}_trans"] == correct_trans
        )
        blocked_seq["correct_key"].append(correct_key)

    return pd.DataFrame(blocked_seq)


# trans to shop mapping
first_block = data[data.block == 1]
second_block = data[data.block == 2]
shop_to_trans = {}
for i in np.arange(4):
    shop_to_trans[i] = first_block[f"trans{i}_shop"].iloc[0]

for k, v in iter_seq.items():
    d = generate_shaping_blocked_seq(first_block, stim_to_food, shop_to_trans, v)
    d = pd.concat([d, second_block])
    print(d.shape)


(156, 18)
(156, 18)
(156, 18)


In [63]:
d

,stim,correct_key,block,img_folder,trans_folder,key0_trans,key1_trans,key2_trans,key3_trans,shop0_food,shop1_food,shop2_food,shop3_food,trans0_shop,trans1_shop,trans2_shop,trans3_shop,set_size
0,4,2,1,1,1,2,0,1,3,1,3,2,0,1,0,3,2,6
1,4,0,1,1,1,1,0,3,2,1,3,2,0,1,0,3,2,6
2,4,1,1,1,1,2,1,0,3,1,3,2,0,1,0,3,2,6
3,4,2,1,1,1,3,2,1,0,1,3,2,0,1,0,3,2,6
4,4,2,1,1,1,3,0,1,2,1,3,2,0,1,0,3,2,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,0,1,2,1,1,1,3,0,2,2,1,0,3,1,0,3,2,6
152,4,0,2,1,1,3,1,0,2,0,2,1,3,1,0,3,2,6
153,3,2,2,1,1,2,3,0,1,0,3,2,1,1,0,3,2,6
154,5,1,2,1,1,0,2,3,1,1,2,0,3,1,0,3,2,6
